# Contrôles de la version v3 : bandes noires, cartes Grad-CAM et robustesse (notebook 13)

**À lancer seulement quand le notebook 12 est terminé** : ce notebook utilise ses six modèles v3 et leurs prédictions de validation. Il utilise aussi les résultats des notebooks 08 v2, 10 v2 et 11. **Le test n'est pas chargé.**

Trois questions, sur les 953 radios de validation :

1. **Les modèles v2 se servent-ils vraiment des bandes noires ?** Au notebook 11, une grande partie de leurs cartes Grad-CAM tombait dans les bandes noires du padding. Une carte montre où le modèle réagit ; elle ne prouve pas que ces zones causent la décision. On fait donc une **expérience contrôlée** : on remplace une bande en haut et une bande en bas de l'image par du **noir**, ou par du **gris moyen**. Le contenu caché est le même dans les deux cas ; seule la couleur des bandes change. Si le score PNEUMONIA monte avec les bandes noires, le modèle utilise le noir comme indice de pneumonie.
2. **Où regardent les modèles v3 ?** Mêmes cartes qu'au notebook 11 (Grad-CAM et attention spatiale du CBAM), comparées à celles des modèles v2 avec une mesure commune.
3. **La version v3 résiste-t-elle aux perturbations ?** Les 10 perturbations du notebook 10 v2 (luminosité, contraste, bruit) sont appliquées aux modèles v3 et comparées aux résultats v2 déjà enregistrés.

## Protocole fixé avant l'analyse

| Élément | Choix |
|---|---|
| Modèles | 18 : v2 sans augmentation et v2 avec augmentation (notebook 10 v2, fichiers contrôlés par SHA-256), v3 (notebook 12) ; 2 architectures ; graines 42, 123 et 2026 |
| Bandes | Une bande en haut et une en bas, chacune de 10 % de la hauteur de l'image. v2 : à l'intérieur du rectangle de l'image, le padding existant est conservé. v3 : sur l'image entière. Trois entrées : originale, bandes grises (valeur moyenne de l'image), bandes noires (0). La modification est faite sur l'entrée du réseau, de la même façon pour v2 et v3. |
| Mesure des bandes | Score PNEUMONIA = logit PNEUMONIA − logit NORMAL. **Effet du noir** = score avec bandes noires − score avec bandes grises, moyenné sur les radios NORMAL, divisé par l'écart-type du score d'origine du modèle, pour comparer des modèles dont l'échelle des logits diffère. Mesures secondaires : faux positifs, part des radios NORMAL dont le score monte. |
| Cartes | Grad-CAM et régions identiques au notebook 11 (bords = bande de 15 % de chaque côté), recalculées ici pour les 18 modèles ; les cartes v2 doivent retrouver celles du notebook 11. Mesure commune v2/v3 : part de la carte de décision située au centre, parmi la partie située dans l'image (hors padding), divisée par la part de surface du centre. C'est l'**enrichissement du centre** : 1 = carte uniforme. Analyse de sensibilité : même mesure en excluant une marge de 16 pixels au bord de l'image. |
| Robustesse v3 | Les 11 conditions du notebook 10 v2, appliquées à l'image étirée **avant** l'égalisation (comme un changement d'acquisition), puis préparation v3 normale. Mêmes paramètres et même bruit (même graine) que pour v2. |

## Prédictions écrites le 28/09/2026, avant de connaître les résultats du notebook 12

| | Prédiction | Règle de lecture |
|---|---|---|
| P1 | Pour chacun des 6 modèles v2 sans augmentation, les bandes noires augmentent le score PNEUMONIA des radios NORMAL (effet du noir > 0). | confirmée si 6 cas sur 6 ; partielle si 4 ou 5 ; infirmée sinon |
| P2 | L'effet du noir est plus faible, en valeur absolue, pour v3 que pour v2 sans augmentation, à architecture et graine égales. | même règle sur les 6 paires |
| P3 | Pour les décisions PNEUMONIA correctes, l'enrichissement du centre est plus élevé pour v3 que pour v2 sans augmentation, à architecture et graine égales. | même règle sur les 6 paires ; refaite en excluant une marge de 16 pixels au bord de l'image (sensibilité) |
| P4 | Sur les 8 conditions de luminosité et de contraste, la variation moyenne du F1 macro est plus favorable (moins négative) pour v3 que pour v2 sans augmentation, à architecture et graine égales. C'est attendu par construction : l'égalisation neutralise ces changements, à l'arrondi et à l'écrêtage près. Aucune prédiction pour le bruit. | même règle sur les 6 paires |

**Limites :** la validation a déjà servi à choisir les époques : ces analyses sont descriptives, la conclusion viendra du test (notebook 14). Les bandes sont des modifications artificielles, et les modèles v3 n'ont jamais vu de bandes noires. Grad-CAM reste une approximation à 28 × 28, et le « centre » n'est pas une segmentation des poumons. Une case de la carte « voit » environ 22 pixels : près du bord de l'image, une carte v2 reflète aussi la transition avec le padding noir, d'où l'analyse de sensibilité de P3.

**Exécution :** noyau `memoire_cnn`, cellules dans l'ordre, environ 15 à 30 minutes sur CPU. Les résultats de chaque modèle sont sauvegardés : après une interruption, réexécuter toutes les cellules, les modèles déjà traités sont réutilisés. **Garder OneDrive en pause et fermer les CSV du projet dans Excel.** À la fin, un seul fichier est à transmettre : `controles_v3/a_envoyer_notebook_13.zip` (tableaux et figures).

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import time
import zipfile
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
import torch.nn.functional as F
from sklearn import __version__ as version_sklearn
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
DOSSIER_08 = PROJET / "optimisation" / "robustesse_validation_v2"
DOSSIER_10 = PROJET / "optimisation" / "robustesse_augmentation_v2"
DOSSIER_11 = PROJET / "optimisation" / "cartes_attention_v1"
DOSSIER_12 = PROJET / "optimisation" / "v3_sans_raccourci_format"
SORTIE = PROJET / "optimisation" / "controles_v3"
APPAREIL = torch.device("cpu")
LOT, TAILLE, TAILLE_CARTE = 16, 224, 28
BANDE_BORDS = 0.15   # régions « centre » et « bords » des cartes, comme au notebook 11
BANDE_TEST = 0.10    # épaisseur de chaque bande du test (en haut et en bas), fixée avant l'analyse
MARGE_CARTES = 16    # analyse de sensibilité des cartes : pixels exclus au bord de l'image (2 cases Grad-CAM)
GRAINE_BRUIT = 314159
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
NOMS_CLASSES = {0: "NORMAL", 1: "PNEUMONIA"}
GRAINES = [42, 123, 2026]
CONFIGURATIONS = {
    "CNN_classique": {"indice": 2, "lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"indice": 3, "lr": 0.0005, "dropout": 0.3},
}
ARCHITECTURES_NOMS = ["CNN_classique", "CNN_CBAM"]
LIBELLES_MODELES = {"CNN_classique": "CNN classique", "CNN_CBAM": "CNN + CBAM"}
V2_SANS, V2_AVEC, V3 = "v2 sans augmentation", "v2 avec augmentation", "v3 sans raccourci"
VERSIONS_MODELES = [V2_SANS, V2_AVEC, V3]
PRETRAITEMENT_V2 = "L; ImageOps.pad 224 bilineaire noir centre; /255"
PRETRAITEMENTS = {
    V2_SANS: PRETRAITEMENT_V2 + "; aucune augmentation",
    V2_AVEC: PRETRAITEMENT_V2,
    V3: "L; resize 224x224 bilineaire sans padding (etirement); ImageOps.equalize; /255; aucune augmentation",
}
VARIANTES_BANDES = ["originale", "bandes_grises", "bandes_noires"]
# Palette catégorielle validée (daltonisme), identique au notebook 12 : une couleur par version, dans cet ordre.
COULEURS = {V2_SANS: "#2a78d6", V2_AVEC: "#eb6834", V3: "#1baf7a"}
POSITIONS_MODELES = {"CNN_classique": 0, "CNN_CBAM": 1}
DECALAGES = {V2_SANS: -0.22, V2_AVEC: 0.0, V3: 0.22}
VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,
    "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn,
}
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def remplacer(temporaire, chemin, essais=20, attente=1.5):
    # Windows ou OneDrive peuvent verrouiller un fichier quelques secondes : on réessaie.
    for essai in range(essais):
        try:
            temporaire.replace(chemin)
            return
        except PermissionError:
            if essai == essais - 1:
                raise PermissionError(f"Impossible d'écrire {chemin} : fermer ce fichier (Excel ?) "
                                      "ou suspendre OneDrive, puis relancer la cellule.")
            time.sleep(attente)

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(objet, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    remplacer(temporaire, chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g")
    remplacer(temporaire, chemin)

def ecrire_npz(chemin, **tableaux):
    temporaire = chemin.with_suffix(".tmp")
    with open(temporaire, "wb") as fichier:
        np.savez_compressed(fichier, **tableaux)
    remplacer(temporaire, chemin)

def enregistrer_figure(figure, chemin):
    temporaire = chemin.with_suffix(".tmp")
    figure.savefig(temporaire, format="png", dpi=160, bbox_inches="tight")
    remplacer(temporaire, chemin)

def nom_fichier(version, nom, graine):
    prefixe = {V2_SANS: "v2_sans", V2_AVEC: "v2_avec", V3: "v3"}[version]
    return f"{prefixe}__{nom}_graine_{graine}"

def resumer_modeles(tableau, groupes, mesures):
    """Entrée : une ligne par graine. Sortie : moyenne, minimum et maximum sur les graines."""
    resume = tableau.groupby(groupes, sort=False)[mesures].agg(["mean", "min", "max"])
    resume.columns = [f"{mesure}_{statistique}" for mesure, statistique in resume.columns]
    return resume.reset_index()

def styliser(axe):
    axe.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    axe.set_axisbelow(True)
    for cote in ["top", "right"]:
        axe.spines[cote].set_visible(False)

def points_par_graine(axe, tableau, colonne):
    """Un point par graine et un trait pour la moyenne ; couleur = version, position = architecture."""
    for version in VERSIONS_MODELES:
        for nom, x in POSITIONS_MODELES.items():
            y = tableau[(tableau["version"] == version) & (tableau["modele"] == nom)][colonne].to_numpy(dtype=float)
            y = y[np.isfinite(y)]
            if len(y) == 0:
                continue
            position = x + DECALAGES[version]
            axe.scatter(np.full(len(y), position), y, s=64, color=COULEURS[version], edgecolors="white",
                        linewidths=1.5, zorder=3, label=version if nom == "CNN_classique" else None)
            axe.hlines(y.mean(), position - 0.08, position + 0.08, color=COULEURS[version], linewidth=2, zorder=2)
    axe.set_xticks(list(POSITIONS_MODELES.values()), [LIBELLES_MODELES[m] for m in POSITIONS_MODELES])
    axe.set_xlim(-0.5, 1.5)
    styliser(axe)

for dossier in [SORTIE, SORTIE / "bandes", SORTIE / "cartes", SORTIE / "robustesse_v3", SORTIE / "figures"]:
    dossier.mkdir(parents=True, exist_ok=True)
print("Appareil :", APPAREIL, "| PyTorch :", torch.__version__)
print("Résultats dans :", SORTIE)

## 1. Préparer la validation et recharger les 18 modèles

Les 953 radios de validation sont préparées de deux façons, et chaque préparation doit avoir exactement la même empreinte SHA-256 qu'avant :

- **v2** : proportions conservées et bandes noires, comme aux notebooks 08 v2 à 11 ;
- **v3** : image étirée en 224 × 224 puis égalisée, comme au notebook 12. L'image étirée avant égalisation est aussi gardée : elle sert au test de robustesse.

Les 12 modèles v2 sont ceux du notebook 10 v2 (fichiers contrôlés par empreinte) ; les 6 modèles v3 sont les fichiers `meilleur.pt` du notebook 12. Pendant les calculs, chaque modèle doit retrouver, sur les images d'origine, les scores bruts déjà enregistrés (notebooks 08 v2, 10 v2 et 12). C'est la preuve que les bons modèles et les bonnes images sont utilisés.

In [ ]:
def preparer_v2(image):
    """Préparation v2 des notebooks 08 à 11 : proportions conservées, bandes noires, masque du rectangle de l'image."""
    gris = image.convert("L")
    preparee = ImageOps.pad(gris, (TAILLE, TAILLE), method=Image.Resampling.BILINEAR, color=0, centering=(0.5, 0.5))
    # Toute l'image source est marquée, même ses pixels initialement noirs.
    blanc = Image.new("L", gris.size, color=255)
    masque = ImageOps.pad(blanc, (TAILLE, TAILLE), method=Image.Resampling.NEAREST, color=0, centering=(0.5, 0.5))
    return np.array(preparee, dtype=np.uint8), np.array(masque) > 0

def etirer(image):
    """Première étape v3 (notebook 12) : niveaux de gris, étirement en 224 × 224 sans bandes noires."""
    return np.array(image.convert("L").resize((TAILLE, TAILLE), Image.Resampling.BILINEAR), dtype=np.uint8)

def egaliser(tableau_u8):
    """Seconde étape v3 (notebook 12) : égalisation d'histogramme."""
    return np.array(ImageOps.equalize(Image.fromarray(np.ascontiguousarray(tableau_u8, dtype=np.uint8))), dtype=np.uint8)

def bornes_regions(masque):
    """Rectangle de l'image et zone centrale, en indices inclus (identique au notebook 11)."""
    lignes = np.flatnonzero(masque.any(axis=1))
    colonnes = np.flatnonzero(masque.any(axis=0))
    exiger(len(lignes) > 0 and len(colonnes) > 0, "Rectangle d'image vide.")
    r0, r1, c0, c1 = int(lignes[0]), int(lignes[-1]), int(colonnes[0]), int(colonnes[-1])
    exiger(bool(masque[r0:r1 + 1, c0:c1 + 1].all()) and int(masque.sum()) == (r1 - r0 + 1) * (c1 - c0 + 1),
           "La zone de l'image n'est pas un rectangle plein.")
    bande_h = int(round(BANDE_BORDS * (r1 - r0 + 1)))
    bande_l = int(round(BANDE_BORDS * (c1 - c0 + 1)))
    cr0, cr1, cc0, cc1 = r0 + bande_h, r1 - bande_h, c0 + bande_l, c1 - bande_l
    exiger(cr0 <= cr1 and cc0 <= cc1, "Zone centrale vide.")
    exiger(bande_h > 0 or bande_l > 0, "Zone des bords vide.")
    return {"r0": r0, "r1": r1, "c0": c0, "c1": c1, "cr0": cr0, "cr1": cr1, "cc0": cc0, "cc1": cc1}

def aires(b):
    """Part de la surface 224 × 224 occupée par chaque région."""
    rectangle = (b["r1"] - b["r0"] + 1) * (b["c1"] - b["c0"] + 1)
    centre = (b["cr1"] - b["cr0"] + 1) * (b["cc1"] - b["cc0"] + 1)
    total = TAILLE * TAILLE
    return {"aire_centre": centre / total, "aire_bords": (rectangle - centre) / total, "aire_padding": (total - rectangle) / total}

def sommes_regions(carte, b):
    """Somme d'une carte 224 × 224 dans le centre, les bords et le padding."""
    total = float(carte.sum())
    rectangle = float(carte[b["r0"]:b["r1"] + 1, b["c0"]:b["c1"] + 1].sum())
    centre = float(carte[b["cr0"]:b["cr1"] + 1, b["cc0"]:b["cc1"] + 1].sum())
    return centre, rectangle - centre, total - rectangle

def signature(modeles):
    return {(m["variante"], m["modele"], int(m["graine"]), m["sha256_sauvegarde"]) for m in modeles}

# Les notebooks 10 v2, 11 et 12 doivent être terminés.
exiger((DOSSIER_10 / "termine.json").is_file(), "Le notebook 10 v2 n'est pas terminé.")
exiger((DOSSIER_11 / "termine.json").is_file() and (DOSSIER_11 / "cartes_par_image.csv").is_file(),
       "Le notebook 11 n'est pas terminé : cartes_par_image.csv est absent.")
exiger((DOSSIER_12 / "resultats_v3.csv").is_file(), "Le notebook 12 n'est pas terminé : resultats_v3.csv est absent.")
PROTOCOLE_10 = json.loads((DOSSIER_10 / "protocole.json").read_text(encoding="utf-8"))
exiger(PROTOCOLE_10.get("version_analyse") == "comparaison_robustesse_augmentation_v2", "Protocole du notebook 10 v2 inattendu.")
exiger(PROTOCOLE_10.get("configurations") == CONFIGURATIONS, "Configurations différentes de celles du notebook 10 v2.")
PROTOCOLE_11 = json.loads((DOSSIER_11 / "protocole.json").read_text(encoding="utf-8"))
exiger(PROTOCOLE_11.get("bande_bords") == BANDE_BORDS and signature(PROTOCOLE_11["modeles"]) == signature(PROTOCOLE_10["modeles"]),
       "Les cartes du notebook 11 ne proviennent pas des mêmes modèles ou des mêmes régions que prévu.")

val_df = pd.read_csv(REPARTITION / "val.csv", dtype={"groupe": str, "empreinte": str})
exiger(val_df["classe"].value_counts().to_dict() == {"NORMAL": 269, "PNEUMONIA": 684}, "Effectifs de validation inattendus.")
exiger(val_df["chemin_relatif"].is_unique, "Chemins dupliqués dans la validation.")
SHA_VAL = empreinte_fichier(REPARTITION / "val.csv")
exiger(SHA_VAL == PROTOCOLE_10["sha256_val_csv"], "La répartition de validation diffère de celle du notebook 10 v2.")
IDENTIFIANTS = val_df["chemin_relatif"].tolist()
REELS = val_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
N_VAL = len(val_df)

RESULTATS_V3 = pd.read_csv(DOSSIER_12 / "resultats_v3.csv")
exiger(len(RESULTATS_V3) == 6 and set(zip(RESULTATS_V3["modele"], RESULTATS_V3["graine"].astype(int)))
       == {(m, g) for m in ARCHITECTURES_NOMS for g in GRAINES},
       "resultats_v3.csv doit contenir les six modèles v3 : laisser le notebook 12 se terminer.")
for ligne in RESULTATS_V3.to_dict("records"):
    reglages = CONFIGURATIONS[ligne["modele"]]
    # Tolérance : la relecture d'un CSV peut modifier le dernier chiffre d'un nombre décimal (0,3 relu 0,29999999999999993).
    exiger(int(ligne["indice"]) == reglages["indice"] and abs(float(ligne["lr"]) - reglages["lr"]) < 1e-12
           and abs(float(ligne["dropout"]) - reglages["dropout"]) < 1e-12,
           f"Réglages v3 inattendus : {ligne['modele']}, graine {ligne['graine']}.")

def dossier_v3(nom, graine):
    return DOSSIER_12 / nom / f"graine_{graine}" / f"configuration_{CONFIGURATIONS[nom]['indice']:02d}"

PROTOCOLES_V3 = {}
for nom in ARCHITECTURES_NOMS:
    for graine in GRAINES:
        fichier = dossier_v3(nom, graine) / "protocole.json"
        exiger(fichier.is_file(), f"Protocole v3 introuvable : {fichier}")
        protocole = json.loads(fichier.read_text(encoding="utf-8"))
        exiger(protocole.get("pretraitement") == PRETRAITEMENTS[V3] and protocole.get("graine") == graine,
               f"Protocole v3 inattendu : {fichier}")
        PROTOCOLES_V3[(nom, graine)] = protocole

PREPAREES = np.empty((N_VAL, TAILLE, TAILLE), dtype=np.uint8)
MASQUES = np.empty((N_VAL, TAILLE, TAILLE), dtype=bool)
ETIREES = np.empty((N_VAL, TAILLE, TAILLE), dtype=np.uint8)
EGALISEES = np.empty((N_VAL, TAILLE, TAILLE), dtype=np.uint8)
LARGEURS = np.empty(N_VAL, dtype=np.int64)
HAUTEURS = np.empty(N_VAL, dtype=np.int64)
BORNES = []
empreinte_v2, empreinte_v3 = hashlib.sha256(), hashlib.sha256()
for i, identifiant in enumerate(tqdm(IDENTIFIANTS, desc="Préparation v2 et v3 de la validation")):
    with Image.open(IMAGES / identifiant) as image:
        LARGEURS[i], HAUTEURS[i] = image.size
        PREPAREES[i], MASQUES[i] = preparer_v2(image)
        ETIREES[i] = etirer(image)
    EGALISEES[i] = egaliser(ETIREES[i])
    exiger(MASQUES[i].any() and (PREPAREES[i][~MASQUES[i]] == 0).all(), "Erreur de géométrie du padding.")
    BORNES.append(bornes_regions(MASQUES[i]))
    empreinte_v2.update(identifiant.encode("utf-8") + b"\0")
    empreinte_v2.update(PREPAREES[i].tobytes())
    empreinte_v2.update(MASQUES[i].tobytes())
    empreinte_v3.update(identifiant.encode("utf-8") + b"\0")
    empreinte_v3.update(EGALISEES[i].tobytes())
SHA_IMAGES_V2, SHA_V3 = empreinte_v2.hexdigest(), empreinte_v3.hexdigest()
exiger(SHA_IMAGES_V2 == PROTOCOLE_10["sha256_images_preparees_et_masques"]
       and SHA_IMAGES_V2 == PROTOCOLE_11["sha256_images_preparees_et_masques"],
       "Les images v2 diffèrent de celles des notebooks 08 à 11.")
for (nom, graine), protocole in PROTOCOLES_V3.items():
    exiger(protocole["sha256_entrees_v3"]["val"] == SHA_V3,
           f"Les entrées v3 diffèrent de celles du notebook 12 ({nom}, graine {graine}).")
RAPPORTS = LARGEURS / HAUTEURS
AIRES = [aires(b) for b in BORNES]
BORNES_V3 = bornes_regions(np.ones((TAILLE, TAILLE), dtype=bool))
AIRES_V3 = aires(BORNES_V3)

def fichier_reference(version, nom, graine):
    """Prédictions déjà enregistrées sur les images d'origine (notebooks 08 v2, 10 v2 et 12)."""
    if version == V2_SANS:
        return DOSSIER_08 / "predictions" / f"{nom}_graine_{graine}__originale.csv"
    if version == V2_AVEC:
        return DOSSIER_10 / "predictions" / f"avec__{nom}_graine_{graine}__originale.csv"
    return DOSSIER_12 / "predictions_validation" / f"{nom}_graine_{graine}.csv"

def logits_reference(version, nom, graine):
    chemin = fichier_reference(version, nom, graine)
    exiger(chemin.is_file(), f"Prédictions de référence introuvables : {chemin}")
    tableau = pd.read_csv(chemin, dtype={"groupe": str})
    exiger(tableau["chemin_relatif"].tolist() == IDENTIFIANTS, f"Ordre des images différent : {chemin}")
    return tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)

REFERENCES = {(v, m, g): logits_reference(v, m, g) for v in VERSIONS_MODELES for m in ARCHITECTURES_NOMS for g in GRAINES}
for ligne in RESULTATS_V3.to_dict("records"):
    logits = REFERENCES[(V3, ligne["modele"], int(ligne["graine"]))]
    f1 = f1_score(REELS, (logits[:, 1] > logits[:, 0]).astype(int), labels=[0, 1], average="macro", zero_division=0)
    exiger(abs(f1 - float(ligne["val_f1_macro"])) < 1e-10,
           f"Prédictions v3 incohérentes avec resultats_v3.csv : {ligne['modele']}, graine {ligne['graine']}.")
print("Validation :", N_VAL, "images. Préparations v2 et v3 identiques à celles des notebooks 08 à 12 (empreintes contrôlées).")
print(f"Rapport largeur/hauteur moyen de l'image d'origine : NORMAL {RAPPORTS[REELS == 0].mean():.2f}, "
      f"PNEUMONIA {RAPPORTS[REELS == 1].mean():.2f}.")
print("Scores de référence réunis pour les 18 modèles.")

In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # Connexion résiduelle, identique aux notebooks 06 v2 à 12.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))

ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
VERSION_DE_VARIANTE = {"sans": V2_SANS, "avec": V2_AVEC}

def charger_modele(version, nom, graine, chemin):
    classe, architecture, nombre_parametres = ARCHITECTURES[nom]
    etat = torch.load(chemin, map_location="cpu", weights_only=True)
    protocole = etat["protocole"]
    exiger(protocole["architecture"] == architecture, f"Architecture différente : {version}, {nom}.")
    exiger(protocole["graine"] == graine, f"Graine incompatible : {version}, {nom}, {graine}.")
    exiger(protocole["classes"] == CLASSES and protocole["pretraitement"] == PRETRAITEMENTS[version],
           f"Classes ou préparation incompatibles : {version}, {nom}, graine {graine}.")
    exiger(etat["configuration"] == CONFIGURATIONS[nom], f"Sauvegarde d'une autre configuration : {version}, {nom}, graine {graine}.")
    if version == V3:
        exiger(protocole["sha256_entrees_v3"]["val"] == SHA_V3, f"Entrées v3 différentes de celles du notebook 12 : {nom}, graine {graine}.")
    with torch.random.fork_rng(devices=[]):
        modele = classe(dropout=CONFIGURATIONS[nom]["dropout"]).to(APPAREIL)
        modele.load_state_dict(etat["modele"], strict=True)
    exiger(sum(p.numel() for p in modele.parameters()) == nombre_parametres, "Nombre de paramètres inattendu.")
    modele.eval()
    modele.requires_grad_(False)
    return modele, int(etat["epoque"])

MODELES, INFORMATIONS = {}, []
exiger(len(PROTOCOLE_10["modeles"]) == 12, "Le protocole du notebook 10 v2 doit lister douze modèles.")
for info in PROTOCOLE_10["modeles"]:
    version, nom, graine = VERSION_DE_VARIANTE[info["variante"]], info["modele"], int(info["graine"])
    chemin = PROJET.joinpath(*info["sauvegarde_relative"].split("/"))
    exiger(chemin.is_file(), f"Sauvegarde introuvable : {chemin}")
    exiger(empreinte_fichier(chemin) == info["sha256_sauvegarde"], f"La sauvegarde {chemin} a changé depuis le notebook 10 v2.")
    modele, epoque = charger_modele(version, nom, graine, chemin)
    exiger(epoque == int(info["epoque"]), "Époque incompatible avec le notebook 10 v2.")
    MODELES[(version, nom, graine)] = modele
    INFORMATIONS.append({"version": version, "modele": nom, "graine": graine, "epoque": epoque,
                         "sha256_sauvegarde": info["sha256_sauvegarde"], "sauvegarde_relative": info["sauvegarde_relative"]})
for ligne in RESULTATS_V3.to_dict("records"):
    nom, graine = ligne["modele"], int(ligne["graine"])
    chemin = dossier_v3(nom, graine) / "meilleur.pt"
    exiger(chemin.is_file(), f"Sauvegarde v3 introuvable : {chemin}")
    modele, epoque = charger_modele(V3, nom, graine, chemin)
    exiger(epoque == int(ligne["epoque_retenue"]), f"Époque v3 incompatible avec resultats_v3.csv : {nom}, graine {graine}.")
    MODELES[(V3, nom, graine)] = modele
    INFORMATIONS.append({"version": V3, "modele": nom, "graine": graine, "epoque": epoque,
                         "sha256_sauvegarde": empreinte_fichier(chemin), "sauvegarde_relative": chemin.relative_to(PROJET).as_posix()})
exiger(set(MODELES) == {(v, m, g) for v in VERSIONS_MODELES for m in ARCHITECTURES_NOMS for g in GRAINES}, "Il manque des modèles.")

def logits_lot(modele, x):
    """Logits (B, 2) pour un lot numpy de forme (B, 1, 224, 224)."""
    with torch.no_grad():
        sortie = modele(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(APPAREIL))
    return sortie.cpu().numpy().astype(np.float64)

class Capture:
    """Conserve la sortie d'une couche pendant le passage avant du modèle."""
    def __init__(self, module):
        self.sortie = None
        self.poignee = module.register_forward_hook(self.enregistrer)
    def enregistrer(self, module, entree, sortie):
        self.sortie = sortie
    def retirer(self):
        self.poignee.remove()

def calculer_lot(nom, modele, x):
    """Logits (B, 2), carte Grad-CAM signée (B, 28, 28) et attention spatiale (B, 28, 28) ou None, pour un lot numpy."""
    couche = modele.cbam if nom == "CNN_CBAM" else modele.bloc3
    capture = Capture(couche)
    capture_spatiale = Capture(modele.cbam.attention_spatiale.convolution) if nom == "CNN_CBAM" else None
    try:
        entree = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(APPAREIL).requires_grad_(True)
        with torch.enable_grad():
            logits = modele(entree)
            activation = capture.sortie
            exiger(activation is not None and tuple(activation.shape) == (x.shape[0], 64, TAILLE_CARTE, TAILLE_CARTE),
                   "La couche utilisée pour Grad-CAM n'a pas la forme attendue.")
            difference = logits[:, 1] - logits[:, 0]
            (gradient,) = torch.autograd.grad(difference.sum(), activation)
        poids = gradient.mean(dim=(2, 3), keepdim=True)
        carte_signee = (poids * activation.detach()).sum(dim=1)
        spatiale = None
        if capture_spatiale is not None:
            sortie = capture_spatiale.sortie
            exiger(sortie is not None and tuple(sortie.shape) == (x.shape[0], 1, TAILLE_CARTE, TAILLE_CARTE),
                   "L'attention spatiale n'a pas la forme attendue.")
            spatiale = torch.sigmoid(sortie.detach())[:, 0].cpu().numpy()
    finally:
        capture.retirer()
        if capture_spatiale is not None:
            capture_spatiale.retirer()
    logits = logits.detach().cpu().numpy().astype(np.float64)
    carte_signee = carte_signee.cpu().numpy()
    exiger(np.isfinite(logits).all() and np.isfinite(carte_signee).all(), "Valeurs non finies dans les logits ou la carte.")
    return logits, carte_signee, spatiale

def agrandir(cartes):
    """Agrandit des cartes (B, 28, 28) en (B, 224, 224) par interpolation bilinéaire."""
    t = torch.from_numpy(np.ascontiguousarray(cartes, dtype=np.float32)).unsqueeze(1)
    return F.interpolate(t, size=(TAILLE, TAILLE), mode="bilinear", align_corners=False)[:, 0].numpy().astype(np.float64)

print("18 modèles chargés et figés : 12 modèles v2 (notebook 10 v2) et 6 modèles v3 (notebook 12).")

In [ ]:
CONDITIONS_ROBUSTESSE = PROTOCOLE_10["conditions"]
IDS_CONDITIONS = [c["id"] for c in CONDITIONS_ROBUSTESSE]
exiger(len(IDS_CONDITIONS) == 11 and IDS_CONDITIONS[0] == "originale" and len(set(IDS_CONDITIONS)) == 11,
       "Les conditions du notebook 10 v2 sont inattendues.")
PREDICTIONS = {
    "P1": "Pour chacun des 6 modèles v2 sans augmentation, les bandes noires augmentent le score PNEUMONIA des radios NORMAL (effet du noir > 0).",
    "P2": "L'effet du noir est plus faible, en valeur absolue, pour v3 que pour v2 sans augmentation (même architecture, même graine).",
    "P3": "Pour les décisions PNEUMONIA correctes, l'enrichissement du centre est plus élevé pour v3 que pour v2 sans augmentation (même architecture, même graine).",
    "P3 sensibilité": "Même comparaison que P3, en excluant une marge de 16 pixels au bord de l'image, pour v2 comme pour v3 (analyse de sensibilité).",
    "P4": "Sur les 8 conditions de luminosité et de contraste, la variation moyenne du F1 macro est plus favorable pour v3 que pour v2 sans augmentation (même architecture, même graine).",
}
REGLE = "confirmée si 6 cas favorables sur 6 ; partielle si 4 ou 5 ; infirmée sinon"

def verdict(favorables, total):
    exiger(total == 6, "Chaque prédiction porte sur 6 cas.")
    if favorables == total:
        return "confirmée"
    return "partielle" if favorables >= 4 else "infirmée"

PROTOCOLE = {
    "version_analyse": "controles_v3", "ensemble": "val", "effectif": N_VAL, "lot": LOT, "taille": TAILLE,
    "test_bandes": {
        "bande_test": BANDE_TEST,
        "position": "une bande en haut et une en bas; epaisseur = round(bande_test x hauteur du rectangle image); v2 : dans le rectangle image, padding conserve; v3 : image entiere",
        "variantes": {"originale": "entree non modifiee", "bandes_grises": "moyenne float64 du rectangle image de l'entree",
                      "bandes_noires": "0"},
        "niveau": "entree du reseau (apres preparation), identique pour v2 et v3",
        "mesure_principale": "moyenne sur les radios NORMAL de (score bandes noires - score bandes grises) / ecart-type du score d'origine du modele; score = logit PNEUMONIA - logit NORMAL",
    },
    "gradcam": {"couche": "sortie de bloc3 (classique) ou du CBAM residuel (CBAM), 64x28x28",
                "score": "logit PNEUMONIA - logit NORMAL; poids = moyenne spatiale du gradient",
                "decision": "ReLU de la carte signee (PNEUMONIA predit) ou de son oppose (NORMAL predit)",
                "agrandissement": "interpolation bilineaire vers 224x224, align_corners=False"},
    "regions": {"bande_bords": BANDE_BORDS, "v2": "padding, bords, centre (notebook 11)", "v3": "bords et centre de l'image entiere"},
    "cartes": "recalculees pour les 18 modeles; cartes v2 controlees contre cartes_par_image.csv du notebook 11",
    "comparaison_cartes": "enrichissement du centre = masse centre / (masse centre + masse bords), divise par aire centre / (aire centre + aire bords)",
    "sensibilite_cartes": {"marge_pixels": MARGE_CARTES, "definition": "meme enrichissement, bords remplaces par les bords prives d'une marge au bord de l'image (sans empieter sur le centre)"},
    "robustesse_v3": {"conditions": CONDITIONS_ROBUSTESSE, "graine_bruit": GRAINE_BRUIT,
                      "application": "image etiree /255 avant egalisation; transformation du notebook 10 v2 sur toute l'image; ecretage [0,1]; arrondi a l'entier 0-255; egalisation; /255",
                      "comparaison_v2": "comparaison_robustesse_augmentation.csv du notebook 10 v2"},
    "predictions_ecrites_le_28_09_2026": PREDICTIONS, "regle_de_lecture": REGLE,
    "sha256_val_csv": SHA_VAL, "sha256_images_v2": SHA_IMAGES_V2, "sha256_entrees_v3_val": SHA_V3,
    "modeles": INFORMATIONS, "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
    "limite": "Validation deja utilisee pour choisir les epoques; bandes artificielles; modeles v3 jamais exposes a des bandes noires; Grad-CAM a 28x28; centre = rectangle fixe; aucune donnee de test.",
}
ID_PROTOCOLE = hashlib.sha256(json.dumps(PROTOCOLE, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE,
           "Le protocole ou les fichiers ont changé. Conserver le dossier existant et choisir un autre nom SORTIE.")
else:
    ecrire_json(PROTOCOLE, fichier_protocole)
BILAN = {}
print("Protocole enregistré :", fichier_protocole)

## 2. Test direct des bandes noires

Pour chaque modèle et chaque radio de validation, trois entrées sont présentées au réseau (figure ci-dessous) :

- **originale** : l'entrée habituelle ;
- **bandes grises** : une bande en haut et une bande en bas de l'image, chacune de 10 % de sa hauteur, remplies avec la valeur moyenne de l'image ;
- **bandes noires** : les mêmes bandes, remplies de noir.

Les bandes grises et les bandes noires cachent exactement le même contenu. La **différence de score entre bandes noires et bandes grises** mesure donc l'effet du noir lui-même. En v2, les bandes sont placées à l'intérieur du rectangle de l'image et le padding existant est conservé ; en v3, elles couvrent le haut et le bas de l'image entière.

Sur l'entrée originale, chaque modèle doit retrouver les scores bruts déjà enregistrés. Un fichier par modèle est écrit dans `bandes/` ; après une interruption, les modèles terminés sont réutilisés.

In [ ]:
def entree_v2(i, variante="originale"):
    """Entrée v2 du réseau ; variantes : bandes grises ou noires en haut et en bas du rectangle de l'image."""
    x = PREPAREES[i].astype(np.float32) / 255.0
    if variante == "originale":
        return x
    b = BORNES[i]
    epaisseur = max(1, int(round(BANDE_TEST * (b["r1"] - b["r0"] + 1))))
    colonnes = slice(b["c0"], b["c1"] + 1)
    valeur = 0.0 if variante == "bandes_noires" else float(x[b["r0"]:b["r1"] + 1, colonnes].mean(dtype=np.float64))
    resultat = x.copy()
    resultat[b["r0"]:b["r0"] + epaisseur, colonnes] = valeur
    resultat[b["r1"] - epaisseur + 1:b["r1"] + 1, colonnes] = valeur
    return resultat

def entree_v3(i, variante="originale"):
    """Entrée v3 du réseau (image étirée et égalisée) ; mêmes variantes, sur l'image entière."""
    x = EGALISEES[i].astype(np.float32) / 255.0
    if variante == "originale":
        return x
    epaisseur = max(1, int(round(BANDE_TEST * TAILLE)))
    valeur = 0.0 if variante == "bandes_noires" else float(x.mean(dtype=np.float64))
    resultat = x.copy()
    resultat[:epaisseur] = valeur
    resultat[TAILLE - epaisseur:] = valeur
    return resultat

ENTREES = {V2_SANS: entree_v2, V2_AVEC: entree_v2, V3: entree_v3}

def lot(version, indices, variante="originale"):
    return np.stack([ENTREES[version](i, variante) for i in indices])[:, None]

def controler_logits(calcules, attendus, contexte):
    ecart = float(np.abs(np.asarray(calcules, dtype=np.float64) - attendus).max())
    tolerance = 1e-4 + 1e-5 * float(np.abs(attendus).max())
    exiger(ecart <= tolerance, f"{contexte} : scores bruts différents des prédictions de référence (écart maximal {ecart:.2e}). "
                               "Vérifier le noyau, les versions et les sauvegardes avant de continuer.")
    return ecart

def lire_existant(fichier, lignes_attendues):
    """Résultats déjà calculés avec ce protocole, sinon None."""
    if not fichier.exists():
        return None
    tableau = pd.read_csv(fichier, dtype={"groupe": str})
    exiger(len(tableau) == lignes_attendues and (tableau["id_protocole"] == ID_PROTOCOLE).all(),
           f"{fichier.name} : résultats existants incompatibles avec ce protocole. Choisir un autre dossier SORTIE.")
    return tableau

def traiter_bandes(version, nom, graine):
    cle = nom_fichier(version, nom, graine)
    fichier = SORTIE / "bandes" / f"{cle}.csv"
    existant = lire_existant(fichier, N_VAL)
    if existant is not None:
        return existant, True
    modele, reference = MODELES[(version, nom, graine)], REFERENCES[(version, nom, graine)]
    logits = {v: np.empty((N_VAL, 2), dtype=np.float64) for v in VARIANTES_BANDES}
    for debut in tqdm(range(0, N_VAL, LOT), desc=cle, leave=False):
        indices = np.arange(debut, min(debut + LOT, N_VAL))
        for variante in VARIANTES_BANDES:
            logits[variante][indices] = logits_lot(modele, lot(version, indices, variante))
        if debut == 0:
            controler_logits(logits["originale"][indices], reference[indices], f"{cle}, premier lot")
    ecart = controler_logits(logits["originale"], reference, cle)
    tableau = val_df[["chemin_relatif", "groupe"]].copy()
    tableau.insert(0, "graine", graine)
    tableau.insert(0, "modele", nom)
    tableau.insert(0, "version", version)
    tableau["classe_vraie"] = REELS
    tableau["rapport_largeur_hauteur"] = RAPPORTS
    for variante in VARIANTES_BANDES:
        tableau[f"score_{variante}"] = logits[variante][:, 1] - logits[variante][:, 0]
        tableau[f"prediction_{variante}"] = (logits[variante][:, 1] > logits[variante][:, 0]).astype(int)
    tableau["ecart_max_logits_reference"] = ecart
    tableau["id_protocole"] = ID_PROTOCOLE
    ecrire_csv(tableau, fichier)
    return tableau, False

# Figure : les trois entrées pour une radio fixée à l'avance (première radio NORMAL de la validation).
i0 = int(np.flatnonzero(REELS == 0)[0])
fig, axes = plt.subplots(2, 3, figsize=(9.6, 7.0), constrained_layout=True)
for rang, (version, libelle) in enumerate([(V2_SANS, "v2"), (V3, "v3")]):
    for colonne, (variante, titre) in enumerate(zip(VARIANTES_BANDES, ["originale", "bandes grises", "bandes noires"])):
        axes[rang, colonne].imshow(ENTREES[version](i0, variante), cmap="gray", vmin=0, vmax=1)
        axes[rang, colonne].set_title(f"Entrée {libelle} : {titre}", fontsize=10)
        axes[rang, colonne].axis("off")
fig.suptitle(f"Les trois entrées du test, pour une radio NORMAL de la validation ({Path(IDENTIFIANTS[i0]).name})")
enregistrer_figure(fig, SORTIE / "figures" / "variantes_bandes.png")
plt.show()
plt.close(fig)

tableaux = []
for version in VERSIONS_MODELES:
    for nom in ARCHITECTURES_NOMS:
        for graine in GRAINES:
            debut = time.perf_counter()
            tableau, reutilise = traiter_bandes(version, nom, graine)
            tableaux.append(tableau)
            etat = "réutilisé" if reutilise else f"calculé en {(time.perf_counter() - debut) / 60:.1f} min"
            print(f"{version:<21} | {nom:<13} | graine {graine:>4} | écart aux logits de référence "
                  f"{tableau['ecart_max_logits_reference'].max():.1e} | {etat}")
BANDES = pd.concat(tableaux, ignore_index=True)
exiger(len(BANDES) == 18 * N_VAL, "Résultats du test des bandes incomplets.")
ecrire_csv(BANDES, SORTIE / "bandes_par_image.csv")
print("Test des bandes terminé pour les 18 modèles.")

In [ ]:
def resume_bandes_modele(t):
    normal, pneumonie = t[t["classe_vraie"] == 0], t[t["classe_vraie"] == 1]
    echelle = float(t["score_originale"].std())
    exiger(echelle > 0, "Scores constants : effet non mesurable.")
    effet_normal = normal["score_bandes_noires"] - normal["score_bandes_grises"]
    effet_pneumonie = pneumonie["score_bandes_noires"] - pneumonie["score_bandes_grises"]
    ligne = {"version": t["version"].iat[0], "modele": t["modele"].iat[0], "graine": int(t["graine"].iat[0]),
             "ecart_type_score_origine": echelle,
             "effet_noir_NORMAL_logits": float(effet_normal.mean()),
             "effet_noir_NORMAL_standardise": float(effet_normal.mean()) / echelle,
             "effet_noir_PNEUMONIA_standardise": float(effet_pneumonie.mean()) / echelle,
             "effet_gris_NORMAL_standardise": float((normal["score_bandes_grises"] - normal["score_originale"]).mean()) / echelle,
             "NORMAL_score_en_hausse_pct": 100 * float((effet_normal > 0).mean()),
             "NORMAL_basculant_vers_PNEUMONIA_pct": 100 * float(((normal["prediction_bandes_grises"] == 0)
                                                                 & (normal["prediction_bandes_noires"] == 1)).mean())}
    for variante in VARIANTES_BANDES:
        ligne[f"faux_positifs_pct_{variante}"] = 100 * float((normal[f"prediction_{variante}"] == 1).mean())
        ligne[f"faux_negatifs_pct_{variante}"] = 100 * float((pneumonie[f"prediction_{variante}"] == 0).mean())
        ligne[f"f1_macro_{variante}"] = f1_score(t["classe_vraie"], t[f"prediction_{variante}"], labels=[0, 1],
                                                 average="macro", zero_division=0)
    ligne["effet_noir_faux_positifs_points"] = ligne["faux_positifs_pct_bandes_noires"] - ligne["faux_positifs_pct_bandes_grises"]
    ligne["effet_noir_faux_negatifs_points"] = ligne["faux_negatifs_pct_bandes_noires"] - ligne["faux_negatifs_pct_bandes_grises"]
    return ligne

BANDES_MODELES = pd.DataFrame([resume_bandes_modele(t) for _, t in BANDES.groupby(["version", "modele", "graine"], sort=False)])
MESURES_BANDES = ["effet_noir_NORMAL_standardise", "effet_noir_PNEUMONIA_standardise", "effet_gris_NORMAL_standardise",
                  "NORMAL_score_en_hausse_pct", "faux_positifs_pct_originale", "faux_positifs_pct_bandes_grises",
                  "faux_positifs_pct_bandes_noires", "effet_noir_faux_positifs_points", "effet_noir_faux_negatifs_points",
                  "NORMAL_basculant_vers_PNEUMONIA_pct", "f1_macro_originale", "f1_macro_bandes_grises", "f1_macro_bandes_noires"]
RESUME_BANDES = resumer_modeles(BANDES_MODELES, ["version", "modele"], MESURES_BANDES)
RESUME_BANDES["graines_effet_noir_positif"] = (BANDES_MODELES.groupby(["version", "modele"], sort=False)["effet_noir_NORMAL_standardise"]
                                               .apply(lambda s: int((s > 0).sum())).to_numpy())

COMPARAISON_BANDES = BANDES_MODELES.pivot_table(index=["modele", "graine"], columns="version",
                                                values="effet_noir_NORMAL_standardise").reset_index()
COMPARAISON_BANDES.columns.name = None
COMPARAISON_BANDES = COMPARAISON_BANDES.rename(columns={V2_SANS: "effet_v2_sans", V2_AVEC: "effet_v2_avec", V3: "effet_v3"})
COMPARAISON_BANDES["v3_plus_faible_que_v2_sans"] = COMPARAISON_BANDES["effet_v3"].abs() < COMPARAISON_BANDES["effet_v2_sans"].abs()

v2_sans = BANDES_MODELES[BANDES_MODELES["version"] == V2_SANS]
favorables = int((v2_sans["effet_noir_NORMAL_standardise"] > 0).sum())
BILAN["P1"] = {"prediction": "P1", "enonce": PREDICTIONS["P1"], "cas_favorables": favorables, "cas": len(v2_sans),
               "verdict": verdict(favorables, len(v2_sans))}
favorables = int(COMPARAISON_BANDES["v3_plus_faible_que_v2_sans"].sum())
BILAN["P2"] = {"prediction": "P2", "enonce": PREDICTIONS["P2"], "cas_favorables": favorables, "cas": len(COMPARAISON_BANDES),
               "verdict": verdict(favorables, len(COMPARAISON_BANDES))}

ecrire_csv(BANDES_MODELES, SORTIE / "bandes_par_modele.csv")
ecrire_csv(RESUME_BANDES, SORTIE / "resume_bandes.csv")
ecrire_csv(COMPARAISON_BANDES, SORTIE / "comparaison_bandes_v2_v3.csv")

print("Effet du noir sur les radios NORMAL (en écarts-types du score) et faux positifs (%), moyenne [min ; max] sur 3 graines :")
display(RESUME_BANDES[["version", "modele", "effet_noir_NORMAL_standardise_mean", "effet_noir_NORMAL_standardise_min",
                       "effet_noir_NORMAL_standardise_max", "graines_effet_noir_positif", "effet_gris_NORMAL_standardise_mean",
                       "NORMAL_score_en_hausse_pct_mean", "faux_positifs_pct_bandes_grises_mean",
                       "faux_positifs_pct_bandes_noires_mean", "NORMAL_basculant_vers_PNEUMONIA_pct_mean"]].round(3))
print("Effet du noir sur les radios PNEUMONIA et faux négatifs (points de %) :")
display(RESUME_BANDES[["version", "modele", "effet_noir_PNEUMONIA_standardise_mean",
                       "effet_noir_faux_negatifs_points_mean"]].round(3))
print("Comparaison appariée par architecture et graine (P2) :")
display(COMPARAISON_BANDES.round(3))
display(pd.DataFrame([BILAN["P1"], BILAN["P2"]]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
points_par_graine(axes[0], BANDES_MODELES, "effet_noir_NORMAL_standardise")
axes[0].set_title("Radios NORMAL : effet des bandes noires sur le score PNEUMONIA", fontsize=11)
axes[0].set_ylabel("Score PNEUMONIA : bandes noires − bandes grises\n(en écarts-types du score du modèle)")
points_par_graine(axes[1], BANDES_MODELES, "effet_noir_faux_positifs_points")
axes[1].set_title("Radios NORMAL classées PNEUMONIA : bandes noires − bandes grises", fontsize=11)
axes[1].set_ylabel("Faux positifs (points de %)")
for axe in axes:
    axe.axhline(0, color="#8a8a8a", linewidth=1)
axes[0].legend(fontsize=9, frameon=False, loc="best")
fig.suptitle("Test direct des bandes noires sur la validation (points : graines ; trait : moyenne)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "effet_bandes_noires.png")
plt.show()
plt.close(fig)

## 3. Cartes Grad-CAM des modèles v3 et comparaison avec v2

Mêmes calculs qu'au notebook 11, sur les images d'origine : **carte de décision** Grad-CAM (zones qui soutiennent la classe prédite) et **attention spatiale** du CBAM. Les cartes sont calculées ici pour les 18 modèles, avec le même code : les cartes v2 doivent retrouver celles enregistrées par le notebook 11 (contrôle automatique).

Les images v3 n'ont pas de padding. Pour comparer v2 et v3 avec la même mesure, on ne garde que la partie de la carte située **dans l'image** :

- **part au centre** = masse au centre ÷ (masse au centre + masse sur les bords) ;
- **enrichissement du centre** = cette part ÷ la part de surface du centre dans l'image (1 = carte uniforme ; plus de 1 = carte concentrée au centre).

**Analyse de sensibilité, fixée avant les résultats.** Une case de la carte Grad-CAM (28 × 28) couvre 8 pixels et « voit » environ 22 pixels de l'image. Près du bord de l'image, une carte v2 peut donc refléter la transition avec le padding noir, c'est-à-dire le raccourci testé en section 2. La même mesure est recalculée en excluant une **marge de 16 pixels** (deux cases) au bord de l'image, pour v2 comme pour v3. Si l'écart entre v2 et v3 disparaît avec la marge exclue, il venait surtout de ce bord.

Pour v2, la part de la carte située dans le padding est rappelée à côté ; elle vaut 0 en v3 par construction. Un fichier par modèle est écrit dans `cartes/`.

In [ ]:
def interieur(b):
    """Rectangle de l'image privé d'une marge de MARGE_CARTES pixels à chaque bord, sans empiéter sur le centre."""
    return (min(b["r0"] + MARGE_CARTES, b["cr0"]), max(b["r1"] - MARGE_CARTES, b["cr1"]),
            min(b["c0"] + MARGE_CARTES, b["cc0"]), max(b["c1"] - MARGE_CARTES, b["cc1"]))

def aire_anneau(b):
    """Part de la surface 224 × 224 occupée par les bords, marge exclue."""
    i0, i1, j0, j1 = interieur(b)
    centre = (b["cr1"] - b["cr0"] + 1) * (b["cc1"] - b["cc0"] + 1)
    return ((i1 - i0 + 1) * (j1 - j0 + 1) - centre) / (TAILLE * TAILLE)

def mesures_carte(carte, b):
    """Parts de la masse d'une carte positive : centre, bords, padding (notebook 11) et bords marge exclue (anneau)."""
    centre, bords, padding = sommes_regions(carte, b)
    total = centre + bords + padding
    if not np.isfinite(total) or total <= 1e-12:
        return {"centre": np.nan, "bords": np.nan, "padding": np.nan, "anneau": np.nan, "nulle": True}
    i0, i1, j0, j1 = interieur(b)
    anneau = float(carte[i0:i1 + 1, j0:j1 + 1].sum()) - centre
    return {"centre": centre / total, "bords": bords / total, "padding": padding / total, "anneau": anneau / total, "nulle": False}

def moyennes_attention(carte, b, a):
    """Valeur moyenne de l'attention spatiale dans chaque région."""
    centre, bords, padding = sommes_regions(carte, b)
    surface = TAILLE * TAILLE
    return {"attention_centre": centre / (a["aire_centre"] * surface),
            "attention_bords": bords / (a["aire_bords"] * surface) if a["aire_bords"] > 0 else np.nan,
            "attention_padding": padding / (a["aire_padding"] * surface) if a["aire_padding"] > 0 else np.nan}

def cumuler(sommes, cle, carte):
    if cle not in sommes:
        sommes[cle] = [np.zeros((TAILLE, TAILLE), dtype=np.float64), 0]
    sommes[cle][0] += carte
    sommes[cle][1] += 1

def traiter_cartes(version, nom, graine):
    cle = nom_fichier(version, nom, graine)
    fichier_csv = SORTIE / "cartes" / f"{cle}.csv"
    fichier_npz = SORTIE / "cartes" / f"{cle}.npz"
    existant = lire_existant(fichier_csv, N_VAL) if fichier_npz.exists() else None
    if existant is not None:
        return existant, True
    modele, reference = MODELES[(version, nom, graine)], REFERENCES[(version, nom, graine)]
    logits = np.empty((N_VAL, 2), dtype=np.float64)
    sommes, lignes = {}, []
    for debut in tqdm(range(0, N_VAL, LOT), desc=cle, leave=False):
        indices = np.arange(debut, min(debut + LOT, N_VAL))
        lg, signee, spatiale = calculer_lot(nom, modele, lot(version, indices))
        logits[indices] = lg
        if debut == 0:
            controler_logits(lg, reference[indices], f"{cle}, cartes, premier lot")
        pneumonie = agrandir(np.maximum(signee, 0.0))
        normale = agrandir(np.maximum(-signee, 0.0))
        attention = agrandir(spatiale) if spatiale is not None else None
        for k, i in enumerate(indices):
            b, a = (BORNES_V3, AIRES_V3) if version == V3 else (BORNES[i], AIRES[i])
            prediction = int(lg[k, 1] > lg[k, 0])
            decision = pneumonie[k] if prediction == 1 else normale[k]
            m_decision = mesures_carte(decision, b)
            ligne = {
                "version": version, "modele": nom, "graine": graine, "chemin_relatif": IDENTIFIANTS[i],
                "groupe": val_df["groupe"].iat[i], "classe_vraie": int(REELS[i]), "prediction": prediction,
                "difference_logits": float(lg[k, 1] - lg[k, 0]), "correcte": prediction == int(REELS[i]),
                **a, "aire_anneau": aire_anneau(b),
                "masse_centre_decision": m_decision["centre"], "masse_bords_decision": m_decision["bords"],
                "masse_padding_decision": m_decision["padding"], "masse_anneau_decision": m_decision["anneau"],
                "carte_decision_nulle": m_decision["nulle"],
            }
            if attention is not None:
                ligne.update(moyennes_attention(attention[k], b, a))
                cumuler(sommes, f"attention__{int(REELS[i])}", attention[k])
            else:
                ligne.update({"attention_centre": np.nan, "attention_bords": np.nan, "attention_padding": np.nan})
            if not m_decision["nulle"]:
                cumuler(sommes, f"decision__{int(REELS[i])}", decision / decision.max())
            lignes.append(ligne)
    ecart = controler_logits(logits, reference, f"{cle}, cartes")
    tableau = pd.DataFrame(lignes)
    tableau["ecart_max_logits_reference"] = ecart
    tableau["id_protocole"] = ID_PROTOCOLE
    ecrire_npz(fichier_npz, **{f"{c}__somme": v[0].astype(np.float32) for c, v in sommes.items()},
               **{f"{c}__nombre": np.array(v[1]) for c, v in sommes.items()})
    ecrire_csv(tableau, fichier_csv)
    return tableau, False

tableaux = []
for version in VERSIONS_MODELES:
    for nom in ARCHITECTURES_NOMS:
        for graine in GRAINES:
            debut = time.perf_counter()
            tableau, reutilise = traiter_cartes(version, nom, graine)
            tableaux.append(tableau)
            etat = "réutilisé" if reutilise else f"calculé en {(time.perf_counter() - debut) / 60:.1f} min"
            print(f"{version:<21} | {nom:<13} | graine {graine:>4} | écart aux logits de référence "
                  f"{tableau['ecart_max_logits_reference'].max():.1e} | {etat}")
CARTES_PAR_IMAGE = pd.concat(tableaux, ignore_index=True)
exiger(len(CARTES_PAR_IMAGE) == 18 * N_VAL, "Cartes incomplètes.")
ecrire_csv(CARTES_PAR_IMAGE, SORTIE / "cartes_par_image.csv")
print("Cartes calculées pour les 18 modèles.")

In [ ]:
def booleens(serie):
    valeurs = serie.map({True: True, False: False, "True": True, "False": False})
    exiger(valeurs.notna().all(), "Colonne booléenne inattendue.")
    return valeurs.astype(bool)

def resumer_images(tableau, groupes, mesures):
    """Moyenne par graine, puis moyenne, écart-type, minimum et maximum sur les graines."""
    par_graine = tableau.groupby(groupes + ["graine"], sort=False)[mesures].mean().reset_index()
    resume = par_graine.groupby(groupes, sort=False)[mesures].agg(["mean", "std", "min", "max"])
    resume.columns = [f"{mesure}_{statistique}" for mesure, statistique in resume.columns]
    effectifs = (tableau.groupby(groupes + ["graine"], sort=False).size().rename("n").reset_index()
                 .groupby(groupes, sort=False)["n"].mean().rename("images_par_graine"))
    return resume.join(effectifs).reset_index()

CARTES = CARTES_PAR_IMAGE.copy()
CARTES["correcte"] = booleens(CARTES["correcte"])
CARTES["carte_decision_nulle"] = booleens(CARTES["carte_decision_nulle"])

# Contrôle : les cartes v2 recalculées ici doivent être celles du notebook 11.
cles_image = ["version", "modele", "graine", "chemin_relatif"]
masses_11 = ["masse_centre_decision", "masse_bords_decision", "masse_padding_decision"]
reference_11 = pd.read_csv(DOSSIER_11 / "cartes_par_image.csv", dtype={"groupe": str})
reference_11 = reference_11[reference_11["condition"] == "originale"].copy()
reference_11["version"] = reference_11["variante"].map({"sans": V2_SANS, "avec": V2_AVEC})
controle = CARTES[CARTES["version"] != V3].merge(reference_11[cles_image + masses_11 + ["prediction"]], on=cles_image,
                                                 suffixes=("", "_11"), validate="one_to_one")
exiger(len(controle) == 12 * N_VAL, "Cartes v2 du notebook 11 incomplètes.")
differentes = controle["prediction"] != controle["prediction_11"]
exiger((controle.loc[differentes, "difference_logits"].abs() < 1e-3).all(), "Des prédictions v2 diffèrent de celles du notebook 11.")
comparables = ~differentes & ~controle["carte_decision_nulle"]
ECART_NOTEBOOK_11 = float(max((controle.loc[comparables, m] - controle.loc[comparables, f"{m}_11"]).abs().max() for m in masses_11))
exiger(ECART_NOTEBOOK_11 < 1e-4, f"Les cartes v2 recalculées diffèrent de celles du notebook 11 (écart maximal {ECART_NOTEBOOK_11:.1e}).")
print(f"Cartes v2 identiques à celles du notebook 11 (écart maximal des parts de masse : {ECART_NOTEBOOK_11:.1e}).")

CARTES["carte_decision_nulle"] = CARTES["carte_decision_nulle"].astype(float)
CARTES["classe"] = CARTES["classe_vraie"].map(NOMS_CLASSES)
dans_image = CARTES["masse_centre_decision"] + CARTES["masse_bords_decision"]
CARTES["part_centre_dans_image"] = CARTES["masse_centre_decision"] / dans_image.where(dans_image > 0)
CARTES["enrichissement_centre_image"] = CARTES["part_centre_dans_image"] / (CARTES["aire_centre"] / (CARTES["aire_centre"] + CARTES["aire_bords"]))
sans_marge = CARTES["masse_centre_decision"] + CARTES["masse_anneau_decision"]
CARTES["enrichissement_centre_marge_exclue"] = ((CARTES["masse_centre_decision"] / sans_marge.where(sans_marge > 0))
                                               / (CARTES["aire_centre"] / (CARTES["aire_centre"] + CARTES["aire_anneau"])))
CARTES["part_marge_bord_image"] = CARTES["masse_bords_decision"] - CARTES["masse_anneau_decision"]
CARTES["part_hors_image"] = CARTES["masse_padding_decision"]
CARTES["ecart_attention_centre_bords"] = CARTES["attention_centre"] - CARTES["attention_bords"]

MESURES_ENRICHISSEMENT = {"enrichissement_centre_image": "image entière",
                          "enrichissement_centre_marge_exclue": f"marge de {MARGE_CARTES} px exclue au bord de l'image"}
MESURES_CARTES = ["part_centre_dans_image", "enrichissement_centre_image", "enrichissement_centre_marge_exclue",
                  "part_marge_bord_image", "part_hors_image", "carte_decision_nulle"]
SYNTHESE_CARTES = resumer_images(CARTES, ["version", "modele", "classe", "correcte"], MESURES_CARTES)
ATTENTION_CBAM = resumer_images(CARTES[CARTES["modele"] == "CNN_CBAM"], ["version", "classe"],
                                ["attention_centre", "attention_bords", "ecart_attention_centre_bords"])
PAR_GRAINE_CARTES = (CARTES[CARTES["correcte"]].groupby(["version", "modele", "graine", "classe"], sort=False)
                     [list(MESURES_ENRICHISSEMENT) + ["part_marge_bord_image", "part_hors_image"]].mean().reset_index())

def ecarts_versions(mesure):
    pivot = PAR_GRAINE_CARTES.pivot_table(index=["classe", "modele", "graine"], columns="version", values=mesure, dropna=False).reset_index()
    pivot.columns.name = None
    pivot.insert(0, "mesure", mesure)
    pivot["ecart_v3_moins_v2_sans"] = pivot[V3] - pivot[V2_SANS]
    pivot["ecart_v3_moins_v2_avec"] = pivot[V3] - pivot[V2_AVEC]
    return pivot

def ecarts_architectures(mesure):
    pivot = PAR_GRAINE_CARTES.pivot_table(index=["version", "classe", "graine"], columns="modele", values=mesure, dropna=False).reset_index()
    pivot.columns.name = None
    pivot.insert(0, "mesure", mesure)
    pivot["ecart_CBAM_moins_classique"] = pivot["CNN_CBAM"] - pivot["CNN_classique"]
    return pivot

ECARTS_CARTES = pd.concat([ecarts_versions(m) for m in MESURES_ENRICHISSEMENT], ignore_index=True)
for cle, mesure in [("P3", "enrichissement_centre_image"), ("P3 sensibilité", "enrichissement_centre_marge_exclue")]:
    lignes_p3 = ECARTS_CARTES[(ECARTS_CARTES["mesure"] == mesure) & (ECARTS_CARTES["classe"] == "PNEUMONIA")]
    favorables = int((lignes_p3["ecart_v3_moins_v2_sans"] > 0).sum())
    BILAN[cle] = {"prediction": cle, "enonce": PREDICTIONS[cle], "cas_favorables": favorables, "cas": len(lignes_p3),
                  "verdict": verdict(favorables, len(lignes_p3))}

ARCHITECTURES_CARTES = pd.concat([ecarts_architectures(m) for m in MESURES_ENRICHISSEMENT], ignore_index=True)
RESUME_CBAM_CARTES = (ARCHITECTURES_CARTES.groupby(["mesure", "version", "classe"], sort=False)["ecart_CBAM_moins_classique"]
                      .agg(["mean", "min", "max"]).reset_index())
RESUME_CBAM_CARTES["graines_CBAM_plus_central"] = (ARCHITECTURES_CARTES.groupby(["mesure", "version", "classe"], sort=False)
                                                   ["ecart_CBAM_moins_classique"].apply(lambda s: int((s > 0).sum())).to_numpy())

ecrire_csv(SYNTHESE_CARTES, SORTIE / "synthese_cartes_v2_v3.csv")
ecrire_csv(ATTENTION_CBAM, SORTIE / "attention_CBAM_v2_v3.csv")
ecrire_csv(ECARTS_CARTES, SORTIE / "ecarts_cartes_v2_v3.csv")
ecrire_csv(ARCHITECTURES_CARTES, SORTIE / "CBAM_classique_cartes.csv")
ecrire_csv(RESUME_CBAM_CARTES, SORTIE / "resume_CBAM_classique_cartes.csv")

print("Décisions correctes : enrichissement du centre (image entière, puis marge exclue), part de la carte dans la marge "
      "du bord de l'image et dans le padding (moyenne sur 3 graines) :")
display(SYNTHESE_CARTES[SYNTHESE_CARTES["correcte"]][["version", "modele", "classe", "images_par_graine",
        "enrichissement_centre_image_mean", "enrichissement_centre_marge_exclue_mean", "part_marge_bord_image_mean",
        "part_hors_image_mean"]].round(3))
print("Écart d'enrichissement v3 − v2, apparié par architecture et graine (décisions correctes) :")
display(ECARTS_CARTES.round(3))
display(pd.DataFrame([BILAN["P3"], BILAN["P3 sensibilité"]]))
print("Écart CBAM − classique de l'enrichissement du centre (décisions correctes), apparié par graine :")
display(RESUME_CBAM_CARTES.round(3))
print("Attention spatiale du CBAM (valeur moyenne entre 0 et 1) :")
display(ATTENTION_CBAM[["version", "classe", "attention_centre_mean", "attention_bords_mean",
                        "ecart_attention_centre_bords_mean"]].round(4))

# Figure : cartes moyennes des modèles v3, par classe vraie.
def carte_moyenne(version, nom, type_carte, classe):
    cartes = []
    for graine in GRAINES:
        with np.load(SORTIE / "cartes" / f"{nom_fichier(version, nom, graine)}.npz") as donnees:
            cle = f"{type_carte}__{classe}"
            if f"{cle}__somme" in donnees.files and int(donnees[f"{cle}__nombre"]) > 0:
                cartes.append(donnees[f"{cle}__somme"].astype(np.float64) / int(donnees[f"{cle}__nombre"]))
    return np.mean(cartes, axis=0) if cartes else None

FOND_V3 = {classe: EGALISEES[REELS == classe].mean(axis=0) / 255.0 for classe in [0, 1]}
attention_moyenne = {classe: carte_moyenne(V3, "CNN_CBAM", "attention", classe) for classe in [0, 1]}
valeurs_attention = np.concatenate([c.ravel() for c in attention_moyenne.values() if c is not None])
colonnes_figure = [("CNN_classique", "decision", "CNN classique\nGrad-CAM décision"),
                   ("CNN_CBAM", "decision", "CNN + CBAM\nGrad-CAM décision"),
                   ("CNN_CBAM", "attention", "CNN + CBAM\nattention spatiale")]
fig, axes = plt.subplots(2, 3, figsize=(12.5, 9.0), constrained_layout=True)
image_attention = None
for ligne, classe in enumerate([0, 1]):
    for colonne, (nom, type_carte, titre) in enumerate(colonnes_figure):
        axe = axes[ligne, colonne]
        axe.imshow(FOND_V3[classe], cmap="gray", vmin=0, vmax=1)
        if type_carte == "attention":
            carte = attention_moyenne[classe]
            if carte is not None:
                image_attention = axe.imshow(carte, cmap="viridis", alpha=0.55, vmin=float(valeurs_attention.min()),
                                             vmax=float(valeurs_attention.max()))
        else:
            carte = carte_moyenne(V3, nom, type_carte, classe)
            if carte is not None:
                axe.imshow(carte / carte.max(), cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axe.set_title(f"{titre}\nradios {NOMS_CLASSES[classe]}", fontsize=10)
        axe.axis("off")
if image_attention is not None:
    fig.colorbar(image_attention, ax=axes[:, 2], fraction=0.05, label="attention spatiale moyenne (0 à 1)")
fig.suptitle("Modèles v3 : cartes moyennes sur la validation (images d'origine, 3 graines)\n"
             "Fond : entrée v3 moyenne de la classe. Rouge : zones les plus utilisées pour la décision.")
enregistrer_figure(fig, SORTIE / "figures" / "cartes_v3_moyennes.png")
plt.show()
plt.close(fig)

# Figure : enrichissement du centre, v2 et v3, avec et sans la marge du bord de l'image.
fig, axes = plt.subplots(2, 2, figsize=(14, 9.8))
for rang, (mesure, libelle) in enumerate(MESURES_ENRICHISSEMENT.items()):
    for colonne, classe in enumerate(["NORMAL", "PNEUMONIA"]):
        axe = axes[rang, colonne]
        points_par_graine(axe, PAR_GRAINE_CARTES[PAR_GRAINE_CARTES["classe"] == classe], mesure)
        axe.axhline(1, color="#8a8a8a", linewidth=1, linestyle="--")
        axe.set_title(f"Décisions correctes sur les radios {classe}\n{libelle}", fontsize=11)
        axe.set_ylabel("Enrichissement du centre\n(1 = carte uniforme)")
axes[0, 0].legend(fontsize=9, frameon=False, loc="best")
fig.suptitle("Concentration de la carte de décision au centre de l'image (points : graines ; trait : moyenne)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "enrichissement_centre_v2_v3.png")
plt.show()
plt.close(fig)

In [ ]:
def afficher(axe, image, carte=None, attention=False):
    axe.imshow(image, cmap="gray", vmin=0, vmax=1)
    if carte is not None:
        if attention:
            axe.imshow(carte, cmap="viridis", alpha=0.55, vmin=float(carte.min()), vmax=float(carte.max()))
        else:
            axe.imshow(carte, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    axe.set_xticks([])
    axe.set_yticks([])

def cartes_exemples(version, nom, indices, type_carte):
    """Cartes 224 × 224 moyennées sur les trois graines, et nombre de graines qui prédisent PNEUMONIA."""
    x = lot(version, indices)
    cumul = np.zeros((len(indices), TAILLE, TAILLE))
    votes = np.zeros(len(indices), dtype=int)
    for graine in GRAINES:
        lg, signee, spatiale = calculer_lot(nom, MODELES[(version, nom, graine)], x)
        pneumonie_predite = lg[:, 1] > lg[:, 0]
        votes += pneumonie_predite.astype(int)
        if type_carte == "attention":
            cumul += agrandir(spatiale)
            continue
        choisies = np.where(pneumonie_predite[:, None, None], np.maximum(signee, 0.0), np.maximum(-signee, 0.0))
        cartes = agrandir(choisies)
        maxima = cartes.reshape(len(indices), -1).max(axis=1)[:, None, None]
        cumul += np.divide(cartes, maxima, out=np.zeros_like(cartes), where=maxima > 0)
    return cumul / len(GRAINES), votes

# Exemples fixés à l'avance, comme au notebook 11 : les trois premières radios NORMAL et PNEUMONIA de la validation.
exemples = np.array(list(np.flatnonzero(REELS == 0)[:3]) + list(np.flatnonzero(REELS == 1)[:3]))
COLONNES_EXEMPLES = [
    (V2_SANS, None, "image", "Entrée v2\n(padding noir)"),
    (V2_SANS, "CNN_classique", "decision", "Classique v2 sans augm.\nGrad-CAM décision"),
    (V2_SANS, "CNN_CBAM", "decision", "CBAM v2 sans augm.\nGrad-CAM décision"),
    (V3, None, "image", "Entrée v3\n(étirée, égalisée)"),
    (V3, "CNN_classique", "decision", "Classique v3\nGrad-CAM décision"),
    (V3, "CNN_CBAM", "decision", "CBAM v3\nGrad-CAM décision"),
    (V3, "CNN_CBAM", "attention", "CBAM v3\nattention spatiale"),
]
resultats_exemples = {(version, nom, type_carte): cartes_exemples(version, nom, exemples, type_carte)
                      for version, nom, type_carte, _ in COLONNES_EXEMPLES if nom is not None}
fig, axes = plt.subplots(len(exemples), len(COLONNES_EXEMPLES), figsize=(2.4 * len(COLONNES_EXEMPLES), 2.55 * len(exemples)))
for ligne, i in enumerate(exemples):
    for colonne, (version, nom, type_carte, titre) in enumerate(COLONNES_EXEMPLES):
        axe = axes[ligne, colonne]
        image = lot(version, [i])[0, 0]
        if nom is None:
            afficher(axe, image)
        else:
            cartes, votes = resultats_exemples[(version, nom, type_carte)]
            afficher(axe, image, cartes[ligne], attention=(type_carte == "attention"))
            if type_carte == "decision":
                axe.text(0.03, 0.03, f"PNEUMONIA : {votes[ligne]}/3", transform=axe.transAxes, fontsize=7, color="white",
                         bbox={"facecolor": "black", "alpha": 0.6, "pad": 1.5, "edgecolor": "none"})
        if ligne == 0:
            axe.set_title(titre, fontsize=8)
    axes[ligne, 0].set_ylabel(f"{NOMS_CLASSES[int(REELS[i])]}\n{Path(IDENTIFIANTS[i]).name[:22]}", fontsize=8)
fig.suptitle("Exemples fixés à l'avance (trois premières radios NORMAL et PNEUMONIA de la validation), v2 sans augmentation et v3\n"
             "Cartes moyennées sur 3 graines ; en bas à gauche : nombre de graines qui prédisent PNEUMONIA")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "exemples_v2_v3.png")
plt.show()
plt.close(fig)

## 4. Robustesse de la version v3 aux perturbations du notebook 10 v2

Les 10 perturbations du notebook 10 v2 (luminosité et contraste × 0,80 à × 1,20, bruit σ = 0,01 et 0,03) sont appliquées aux six modèles v3. Chaque perturbation est appliquée à l'image étirée, **avant l'égalisation** : elle représente un changement des conditions d'acquisition, que la préparation v3 traite ensuite normalement. Le bruit est exactement le même qu'en v2 (même graine, même tirage pour chaque image).

Les résultats v2 (sans et avec augmentation) sont repris du fichier `comparaison_robustesse_augmentation.csv` du notebook 10 v2. Pour chaque modèle, on compare la **variation** de chaque mesure par rapport aux images d'origine.

**Attention à l'interprétation :** l'égalisation d'histogramme donne presque la même entrée quand la luminosité ou le contraste changent. Une bonne robustesse de v3 à ces perturbations est donc attendue par construction : c'est une propriété de la préparation, pas du réseau. Le bruit est la perturbation réellement informative.

In [ ]:
CONDITION_PAR_ID = {c["id"]: c for c in CONDITIONS_ROBUSTESSE}
IDS_PERTURBEES = [cid for cid in IDS_CONDITIONS if cid != "originale"]
METRIQUES_ROBUSTESSE = ["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL", "exactitude"]

def bruit_image(identifiant, forme):
    """Bruit gaussien propre à chaque image, identique à celui des notebooks 08 v2 et 10 v2."""
    cle = f"{GRAINE_BRUIT}|{identifiant}".encode("utf-8")
    graine = int.from_bytes(hashlib.sha256(cle).digest()[:8], "little")
    generateur = np.random.Generator(np.random.PCG64(graine))
    return generateur.standard_normal(forme).astype(np.float32)

def entree_v3_perturbee(i, condition):
    """Perturbation de l'image étirée (avant égalisation), comme un changement d'acquisition, puis égalisation v3."""
    nature, valeur = condition["type"], condition["valeur"]
    if nature == "identite":
        return entree_v3(i)
    x = ETIREES[i].astype(np.float32) / 255.0
    if nature == "luminosite":
        x = x * np.float32(valeur)
    elif nature == "contraste":
        moyenne = x.mean(dtype=np.float64)
        x = (x - moyenne) * valeur + moyenne
    elif nature == "bruit":
        x = x + np.float32(valeur) * bruit_image(IDENTIFIANTS[i], x.shape)
    else:
        raise ValueError(f"Transformation inconnue : {nature}")
    brute = np.rint(np.clip(x, 0.0, 1.0) * 255.0).astype(np.uint8)
    return egaliser(brute).astype(np.float32) / 255.0

# Contrôle : sans perturbation, l'aller-retour en [0, 1] puis l'arrondi redonnent exactement l'image étirée.
for i in range(3):
    exiger(np.array_equal(np.rint(np.clip(ETIREES[i].astype(np.float32) / 255.0, 0.0, 1.0) * 255.0).astype(np.uint8), ETIREES[i]),
           "L'arrondi modifie l'image étirée : contrôle impossible.")

def traiter_robustesse_v3(nom, graine):
    cle = nom_fichier(V3, nom, graine)
    fichier = SORTIE / "robustesse_v3" / f"{cle}.csv"
    existant = lire_existant(fichier, N_VAL)
    if existant is not None:
        return existant, True
    modele = MODELES[(V3, nom, graine)]
    logits = {cid: np.empty((N_VAL, 2), dtype=np.float64) for cid in IDS_CONDITIONS}
    for debut in tqdm(range(0, N_VAL, LOT), desc=cle, leave=False):
        indices = np.arange(debut, min(debut + LOT, N_VAL))
        for condition in CONDITIONS_ROBUSTESSE:
            x = np.stack([entree_v3_perturbee(i, condition) for i in indices])[:, None]
            logits[condition["id"]][indices] = logits_lot(modele, x)
    ecart = controler_logits(logits["originale"], REFERENCES[(V3, nom, graine)], f"{cle}, robustesse")
    tableau = val_df[["chemin_relatif", "groupe"]].copy()
    tableau.insert(0, "graine", graine)
    tableau.insert(0, "modele", nom)
    tableau.insert(0, "version", V3)
    tableau["classe_vraie"] = REELS
    for cid in IDS_CONDITIONS:
        tableau[f"score_{cid}"] = logits[cid][:, 1] - logits[cid][:, 0]
    tableau["ecart_max_logits_reference"] = ecart
    tableau["id_protocole"] = ID_PROTOCOLE
    ecrire_csv(tableau, fichier)
    return tableau, False

def mesures(y, score):
    prediction = (score > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {"f1_macro": f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0),
            "auc_roc": roc_auc_score(y, score), "rappel_pneumonie": tp / (tp + fn), "specificite_NORMAL": tn / (tn + fp),
            "exactitude": (tn + tp) / len(y), "faux_positifs": int(fp), "faux_negatifs": int(fn)}

lignes = []
for nom in ARCHITECTURES_NOMS:
    for graine in GRAINES:
        debut = time.perf_counter()
        tableau, reutilise = traiter_robustesse_v3(nom, graine)
        etat = "réutilisé" if reutilise else f"calculé en {(time.perf_counter() - debut) / 60:.1f} min"
        print(f"v3 | {nom:<13} | graine {graine:>4} | écart aux logits de référence "
              f"{tableau['ecart_max_logits_reference'].max():.1e} | {etat}")
        for cid in IDS_CONDITIONS:
            lignes.append({"version": V3, "modele": nom, "graine": graine, "condition": cid, "type": CONDITION_PAR_ID[cid]["type"],
                           **mesures(REELS, tableau[f"score_{cid}"].to_numpy(dtype=np.float64))})

# Résultats v2 du notebook 10 v2 (mêmes modèles, mêmes conditions, mêmes images).
fichier_comparaison_10 = DOSSIER_10 / "comparaison_robustesse_augmentation.csv"
exiger(fichier_comparaison_10.is_file(), f"Fichier du notebook 10 v2 introuvable : {fichier_comparaison_10}")
comparaison_10 = pd.read_csv(fichier_comparaison_10)
exiger(len(comparaison_10) == 6 * len(IDS_CONDITIONS) and set(comparaison_10["condition"]) == set(IDS_CONDITIONS),
       "comparaison_robustesse_augmentation.csv est incomplet.")
for suffixe, version in [("sans", V2_SANS), ("avec", V2_AVEC)]:
    for ligne in comparaison_10.to_dict("records"):
        lignes.append({"version": version, "modele": ligne["modele"], "graine": int(ligne["graine"]), "condition": ligne["condition"],
                       "type": CONDITION_PAR_ID[ligne["condition"]]["type"],
                       **{m: float(ligne[f"{m}_{suffixe}"]) for m in METRIQUES_ROBUSTESSE},
                       "faux_positifs": int(ligne[f"faux_positifs_{suffixe}"]), "faux_negatifs": int(ligne[f"faux_negatifs_{suffixe}"])})
ROBUSTESSE = pd.DataFrame(lignes)
exiger(len(ROBUSTESSE) == 18 * len(IDS_CONDITIONS) and not ROBUSTESSE.duplicated(["version", "modele", "graine", "condition"]).any(),
       "Tableau de robustesse incomplet.")
# Contrôle : le F1 d'origine des modèles v2 doit être celui de leurs prédictions de référence.
for version in [V2_SANS, V2_AVEC]:
    for nom in ARCHITECTURES_NOMS:
        for graine in GRAINES:
            reference = REFERENCES[(version, nom, graine)]
            attendu = mesures(REELS, reference[:, 1] - reference[:, 0])["f1_macro"]
            lu = ROBUSTESSE[(ROBUSTESSE["version"] == version) & (ROBUSTESSE["modele"] == nom) & (ROBUSTESSE["graine"] == graine)
                            & (ROBUSTESSE["condition"] == "originale")]["f1_macro"].iat[0]
            exiger(abs(attendu - lu) < 1e-9, f"F1 d'origine incohérent avec les prédictions de référence : {version}, {nom}, graine {graine}.")
base = ROBUSTESSE[ROBUSTESSE["condition"] == "originale"][["version", "modele", "graine"] + METRIQUES_ROBUSTESSE]
ROBUSTESSE = ROBUSTESSE.merge(base, on=["version", "modele", "graine"], suffixes=("", "_originale"), validate="many_to_one")
for m in METRIQUES_ROBUSTESSE:
    ROBUSTESSE[f"variation_{m}"] = ROBUSTESSE[m] - ROBUSTESSE[f"{m}_originale"]

MESURES_ROBUSTESSE = ["f1_macro", "variation_f1_macro", "specificite_NORMAL", "variation_specificite_NORMAL",
                      "rappel_pneumonie", "variation_rappel_pneumonie", "auc_roc", "variation_auc_roc"]
SYNTHESE_ROBUSTESSE = resumer_modeles(ROBUSTESSE, ["version", "modele", "condition"], MESURES_ROBUSTESSE)
perturbees = ROBUSTESSE[ROBUSTESSE["condition"] != "originale"]
par_type = perturbees.groupby(["version", "modele", "graine", "type"], sort=False)[["variation_f1_macro", "variation_specificite_NORMAL",
                                                                                  "variation_rappel_pneumonie"]].mean().reset_index()
ROBUSTESSE_PAR_TYPE = resumer_modeles(par_type, ["version", "modele", "type"],
                                      ["variation_f1_macro", "variation_specificite_NORMAL", "variation_rappel_pneumonie"])
pires = perturbees.loc[perturbees.groupby(["version", "modele", "graine"], sort=False)["variation_f1_macro"].idxmin()]
PIRE_CONDITION = resumer_modeles(pires, ["version", "modele"], ["variation_f1_macro", "f1_macro"])
PIRE_CONDITION["pires_conditions_par_graine"] = pires.groupby(["version", "modele"], sort=False)["condition"].apply(" ; ".join).to_numpy()

ARCHITECTURES_ROBUSTESSE = ROBUSTESSE.pivot_table(index=["version", "graine", "condition"], columns="modele", values="f1_macro").reset_index()
ARCHITECTURES_ROBUSTESSE.columns.name = None
ARCHITECTURES_ROBUSTESSE["ecart_CBAM_moins_classique"] = ARCHITECTURES_ROBUSTESSE["CNN_CBAM"] - ARCHITECTURES_ROBUSTESSE["CNN_classique"]
RESUME_ARCHITECTURES_ROBUSTESSE = (ARCHITECTURES_ROBUSTESSE.groupby(["version", "condition"])["ecart_CBAM_moins_classique"]
                                   .agg(["mean", "min", "max"]).reset_index())
RESUME_ARCHITECTURES_ROBUSTESSE["graines_CBAM_superieur"] = (ARCHITECTURES_ROBUSTESSE.groupby(["version", "condition"])
                                                             ["ecart_CBAM_moins_classique"].apply(lambda s: int((s > 0).sum())).to_numpy())
rang_version = {v: k for k, v in enumerate(VERSIONS_MODELES)}
rang_condition = {c: k for k, c in enumerate(IDS_CONDITIONS)}
RESUME_ARCHITECTURES_ROBUSTESSE = RESUME_ARCHITECTURES_ROBUSTESSE.sort_values(
    ["version", "condition"], key=lambda s: s.map(rang_version) if s.name == "version" else s.map(rang_condition)).reset_index(drop=True)

LUMINOSITE_CONTRASTE = [c["id"] for c in CONDITIONS_ROBUSTESSE if c["type"] in ("luminosite", "contraste")]
exiger(len(LUMINOSITE_CONTRASTE) == 8, "Il faut 8 conditions de luminosité et de contraste.")
P4 = (ROBUSTESSE[ROBUSTESSE["condition"].isin(LUMINOSITE_CONTRASTE)]
      .groupby(["modele", "graine", "version"])["variation_f1_macro"].mean().unstack("version").reset_index())
P4.columns.name = None
P4["v3_plus_favorable_que_v2_sans"] = P4[V3] > P4[V2_SANS]
favorables = int(P4["v3_plus_favorable_que_v2_sans"].sum())
BILAN["P4"] = {"prediction": "P4", "enonce": PREDICTIONS["P4"], "cas_favorables": favorables, "cas": len(P4),
               "verdict": verdict(favorables, len(P4))}
ORDRE_BILAN = ["P1", "P2", "P3", "P3 sensibilité", "P4"]
exiger(set(BILAN) == set(ORDRE_BILAN), "Exécuter toutes les cellules dans l'ordre pour obtenir le bilan complet des prédictions.")
BILAN_PREDICTIONS = pd.DataFrame([BILAN[p] for p in ORDRE_BILAN])
BILAN_PREDICTIONS["regle"] = REGLE

ecrire_csv(ROBUSTESSE, SORTIE / "robustesse_v2_v3.csv")
ecrire_csv(SYNTHESE_ROBUSTESSE, SORTIE / "synthese_robustesse_v2_v3.csv")
ecrire_csv(ROBUSTESSE_PAR_TYPE, SORTIE / "robustesse_par_type.csv")
ecrire_csv(PIRE_CONDITION, SORTIE / "pire_condition_robustesse.csv")
ecrire_csv(ARCHITECTURES_ROBUSTESSE, SORTIE / "CBAM_classique_robustesse.csv")
ecrire_csv(RESUME_ARCHITECTURES_ROBUSTESSE, SORTIE / "resume_CBAM_classique_robustesse.csv")
ecrire_csv(P4, SORTIE / "comparaison_luminosite_contraste_v2_v3.csv")
ecrire_csv(BILAN_PREDICTIONS, SORTIE / "bilan_predictions.csv")

print("Variation moyenne du F1 macro par type de perturbation (moyenne [min ; max] sur 3 graines) :")
display(ROBUSTESSE_PAR_TYPE[["version", "modele", "type", "variation_f1_macro_mean", "variation_f1_macro_min",
                             "variation_f1_macro_max", "variation_specificite_NORMAL_mean", "variation_rappel_pneumonie_mean"]].round(4))
print("Pire perturbation pour chaque modèle :")
display(PIRE_CONDITION.round(4))
print("F1 macro CBAM − classique en v3, apparié par graine, dans chaque condition :")
display(RESUME_ARCHITECTURES_ROBUSTESSE[RESUME_ARCHITECTURES_ROBUSTESSE["version"] == V3].round(4))
print("Bilan des prédictions écrites avant l'analyse :")
display(BILAN_PREDICTIONS)

etiquettes = [CONDITION_PAR_ID[cid]["libelle"].replace(" ", "\n", 1) for cid in IDS_PERTURBEES]
fig, axes = plt.subplots(2, 1, figsize=(15, 9.5), sharex=True)
for axe, nom in zip(axes, ARCHITECTURES_NOMS):
    for version in VERSIONS_MODELES:
        for k, cid in enumerate(IDS_PERTURBEES):
            y = ROBUSTESSE[(ROBUSTESSE["version"] == version) & (ROBUSTESSE["modele"] == nom)
                           & (ROBUSTESSE["condition"] == cid)]["variation_f1_macro"].to_numpy(dtype=float)
            position = k + DECALAGES[version]
            axe.scatter(np.full(len(y), position), y, s=36, color=COULEURS[version], edgecolors="white", linewidths=1.0,
                        zorder=3, label=version if k == 0 else None)
            axe.hlines(y.mean(), position - 0.08, position + 0.08, color=COULEURS[version], linewidth=2, zorder=2)
    axe.axhline(0, color="#8a8a8a", linewidth=1)
    axe.set_title(LIBELLES_MODELES[nom], fontsize=11)
    axe.set_ylabel("Variation du F1 macro\n(condition − originale)")
    styliser(axe)
axes[1].set_xticks(range(len(IDS_PERTURBEES)), etiquettes, fontsize=9)
fig.suptitle("Robustesse sur la validation : variation du F1 macro sous les perturbations du notebook 10 v2\n"
             "(points : graines ; trait : moyenne ; v3 : perturbation appliquée avant l'égalisation)")
fig.tight_layout()
# Légende sous la figure, pour ne masquer aucun point.
poignees, libelles = axes[0].get_legend_handles_labels()
fig.legend(poignees, libelles, loc="upper center", ncol=3, frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.0))
enregistrer_figure(fig, SORTIE / "figures" / "robustesse_v2_v3.png")
plt.show()
plt.close(fig)

ecrire_json({"id_protocole": ID_PROTOCOLE, "lignes_bandes": int(len(BANDES)), "lignes_cartes": int(len(CARTES_PAR_IMAGE)),
             "ecart_cartes_v2_notebook_11": ECART_NOTEBOOK_11, "lignes_robustesse": int(len(ROBUSTESSE)),
             "bilan": [{"prediction": p, "cas_favorables": int(BILAN[p]["cas_favorables"]), "verdict": BILAN[p]["verdict"]}
                       for p in ORDRE_BILAN]},
            SORTIE / "termine.json")

# Un seul fichier à envoyer : les tableaux utiles et toutes les figures, réunis dans une archive zip.
FICHIERS_A_ENVOYER = ["bilan_predictions.csv", "bandes_par_modele.csv", "resume_bandes.csv", "comparaison_bandes_v2_v3.csv",
                      "synthese_cartes_v2_v3.csv", "ecarts_cartes_v2_v3.csv", "resume_CBAM_classique_cartes.csv",
                      "attention_CBAM_v2_v3.csv", "synthese_robustesse_v2_v3.csv", "robustesse_par_type.csv",
                      "pire_condition_robustesse.csv", "resume_CBAM_classique_robustesse.csv",
                      "comparaison_luminosite_contraste_v2_v3.csv", "termine.json"]
archive = SORTIE / "a_envoyer_notebook_13.zip"
temporaire = archive.with_suffix(".tmp")
with zipfile.ZipFile(temporaire, "w", compression=zipfile.ZIP_DEFLATED) as fichier_zip:
    for nom_tableau in FICHIERS_A_ENVOYER:
        fichier_zip.write(SORTIE / nom_tableau, arcname=nom_tableau)
    for figure in sorted((SORTIE / "figures").glob("*.png")):
        fichier_zip.write(figure, arcname=f"figures/{figure.name}")
remplacer(temporaire, archive)
print("Terminé. Pour transmettre les résultats, un seul fichier suffit (tableaux et figures) :")
print(" ", archive)

## Lecture des résultats

**Bilan des prédictions.** Le fichier `bilan_predictions.csv` applique les règles écrites en tête du notebook. Une prédiction infirmée est un résultat à rapporter tel quel dans le mémoire.

**Bandes noires (P1, P2)**

- Un effet du noir positif pour les modèles v2 est une **preuve par intervention** : à contenu caché identique, le noir des bandes pousse le score vers PNEUMONIA. C'est plus fort que les cartes du notebook 11, qui montrent seulement où les modèles réagissent. Le test prouve que les modèles utilisent le noir des bandes ; le lien avec le format d'origine des radios reste une interprétation, appuyée par le diagnostic du notebook 11.
- `NORMAL_basculant_vers_PNEUMONIA_pct` traduit cet effet en erreurs : la part des radios NORMAL qui deviennent des faux positifs quand les bandes grises deviennent noires.
- `effet_gris_NORMAL_standardise` mesure l'effet du simple masquage (bandes grises contre image d'origine). Il sert de repère pour juger la taille de l'effet du noir.
- Un effet proche de 0 en v3 indique que ce raccourci a disparu avec la nouvelle préparation. Un effet encore net en v3 signifierait que le réseau réagit aux zones très sombres pour une autre raison.

**Cartes (P3)**

- Un enrichissement du centre plus élevé en v3 indique que les décisions s'appuient davantage sur le centre de l'image, où se trouvent les poumons.
- Comparer P3 et « P3 sensibilité ». Si les deux vont dans le même sens, la conclusion ne dépend pas du bord de l'image. Si l'écart v3 − v2 n'existe qu'avec l'image entière, il venait surtout du bord de l'image v2, c'est-à-dire de la transition avec le padding : c'est alors une autre trace du raccourci testé en P1, pas la preuve que v3 regarde davantage les poumons. Si les deux mesures se contredisent autrement, P3 ne permet pas de conclure. La colonne `part_marge_bord_image` montre directement la part de la carte située dans la marge du bord.
- L'écart CBAM − classique en v3 est l'argument d'interprétabilité du mémoire, cette fois sans le raccourci du padding. Un CBAM plus concentré au centre sans gain de F1 reste un résultat utile : l'attention peut rendre les cartes plus plausibles sans améliorer la classification.

**Robustesse (P4)**

- Pour la luminosité et le contraste, la robustesse de v3 doit être présentée comme un effet de la préparation (égalisation), pas comme une robustesse apprise par le réseau.
- Le bruit est le vrai test : l'égalisation peut amplifier le bruit dans les zones de faible contraste.
- L'écart CBAM − classique sous perturbation, en v3, complète la question principale du mémoire.

**Limites**

- La validation a déjà servi à choisir les époques : ces résultats sont descriptifs. Les conclusions du mémoire reposeront sur le test.
- Les bandes sont des modifications artificielles, d'épaisseur fixée à l'avance (10 %). Les modèles v3 n'ont jamais vu de bandes noires pendant l'apprentissage.
- Grad-CAM reste une approximation à 28 × 28 ; le « centre » est un rectangle fixe, pas une segmentation des poumons.
- Avec 3 graines par version, la règle « 6 cas sur 6 » décrit la cohérence des résultats ; ce n'est pas un test statistique.

**Suite :** envoyer le fichier `a_envoyer_notebook_13.zip` (tableaux et figures). Le notebook 14 fera ensuite l'évaluation finale sur le test, une seule fois, avec les hypothèses H1 à H4 fixées dans le notebook 12.